In [1]:
%matplotlib inline

In [2]:
#Import your libraries here

import numpy as np
import pandas as pd
import scipy.optimize as sco
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import itertools
import os
import sys
from pathlib import Path

In [3]:
notebook_path = Path(os.getcwd()).resolve()

def get_root(path):
    for parent in [path] + list(path.parents):
        if (parent / "static_data").exists():
            return parent
    return path

PROJECT_ROOT = get_root(notebook_path)
DATA_DIR = PROJECT_ROOT / "static_data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Source:  {DATA_DIR}")

Project Root: D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and_ML_upskill_program\Data_Science\08_final_project
Data Source:  D:\Users\kamen.dimitrov\Desktop\SOFTUNI\AI_and_ML_upskill_program\Data_Science\08_final_project\static_data


In [4]:
import importlib

BASE_URL = "https://storage.googleapis.com/softuni_data_science_final_project_kamend1/static_data/"

from src.data_pipeline_utils import data_fetching_handling as data_pipe
import src.plotting_utils.plotting_utils as plot_utils
import src.signal_testing_utils.portfolio_testing_utils as pf_test
importlib.reload(plot_utils)
importlib.reload(data_pipe)

<module 'src.data_pipeline_utils.data_fetching_handling' from 'D:\\Users\\kamen.dimitrov\\Desktop\\SOFTUNI\\AI_and_ML_upskill_program\\Data_Science\\08_final_project\\src\\data_pipeline_utils\\data_fetching_handling.py'>

# #TODO

text description

In [5]:
tickers = [
    "AAPL", "GOOG", "MSFT", "NVDA",
    "JPM", "BAC", "F", "UPS", "WMT", "TGT",
    "VZ", "T", "FE", "PFE", "JNJ", "DIS",
    "V", "MCD", "NKE", "XOM", "CVX",
    "CAT", "DE", "LMT", "AMD", "INTC", "ORCL", 
    "CRM", "CB", "PG"
]
returns_df = pd.read_csv(BASE_URL + "stock_returns_static_dataset.csv")
returns_df["Date"] = pd.to_datetime(returns_df["Date"])

test_returns_df = returns_df[returns_df["Date"] <= "2021-03-31"].copy()
test_returns_df = test_returns_df.set_index("Date").sort_index()

target_returns_df = returns_df[returns_df["Date"] >= "2021-04-01"].copy()
target_returns_df = target_returns_df.set_index("Date").sort_index()

print(test_returns_df.shape, target_returns_df.shape)
print(test_returns_df.index.min(), test_returns_df.index.max())
print(target_returns_df.index.min(), target_returns_df.index.max())

(1259, 30) (1254, 30)
2016-04-01 00:00:00 2021-03-31 00:00:00
2021-04-01 00:00:00 2026-03-30 00:00:00


In [6]:
trading_days = 252
risk_free_rate = 0.03
mean_returns = test_returns_df.mean() * trading_days
cov_matrix = test_returns_df.cov() * trading_days

In [7]:
number_of_stocks = len(tickers)
weights = np.array([1 / number_of_stocks] * number_of_stocks)

equal_weight_portfolio = list(zip(tickers, weights))

returns = np.dot(weights, mean_returns)
std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
print(returns, std_dev)

0.18086381875792062 0.19331943148184835


In [8]:
args = (mean_returns, cov_matrix)

# Constraint: sum of weights equals 1
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

# Bounds: weights between 0 and 1
bounds = tuple((0, 1) for asset in range(number_of_stocks))

# Initial guess (equal distribution)
initial_guess = number_of_stocks * [1. / number_of_stocks]

In [9]:
def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.03):
    p_ret = np.dot(weights, mean_returns)
    p_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return -(p_ret - risk_free_rate) / p_std

In [10]:
optimal_portfolio = sco.minimize(
    negative_sharpe_ratio, 
    initial_guess, 
    args=args,
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

optimal_weights = optimal_portfolio.x
optimal_return = np.sum(mean_returns * optimal_weights)
optimal_variance = np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights))
optimal_volatility = np.sqrt(optimal_variance)
optimal_sharpe_ratio = optimal_return / optimal_volatility

print(f"Expected Return: {optimal_return:.4f}")
print(f"Variance: {optimal_variance:.4f}")

optimal_portfolio_metrics = {"Expected Return": [optimal_return],
                               "Volatility": [optimal_volatility],}

pd.DataFrame(optimal_portfolio_metrics) \
    .to_csv(DATA_DIR / "optimal_portfolio_metrics.csv", index=False)

Expected Return: 0.3998
Variance: 0.0763


In [11]:
optimal_portfolio = list(zip(tickers, optimal_weights))

opt_series = pd.Series(dict(optimal_portfolio), name="Optimal")
eq_series  = pd.Series(dict(equal_weight_portfolio), name="Equal")

weights_df = pd.concat([opt_series, eq_series], axis=1)

weights_df = weights_df.sort_values("Optimal", ascending=False)

print(weights_df.round(4))

      Optimal   Equal
DE     0.3239  0.0333
MSFT   0.1835  0.0333
AMD    0.1750  0.0333
NVDA   0.1516  0.0333
WMT    0.0800  0.0333
AAPL   0.0484  0.0333
TGT    0.0376  0.0333
JPM    0.0000  0.0333
CRM    0.0000  0.0333
ORCL   0.0000  0.0333
XOM    0.0000  0.0333
UPS    0.0000  0.0333
LMT    0.0000  0.0333
F      0.0000  0.0333
PG     0.0000  0.0333
CAT    0.0000  0.0333
FE     0.0000  0.0333
PFE    0.0000  0.0333
GOOG   0.0000  0.0333
T      0.0000  0.0333
VZ     0.0000  0.0333
BAC    0.0000  0.0333
CVX    0.0000  0.0333
NKE    0.0000  0.0333
V      0.0000  0.0333
MCD    0.0000  0.0333
DIS    0.0000  0.0333
JNJ    0.0000  0.0333
INTC   0.0000  0.0333
CB     0.0000  0.0333


In [24]:
optimal_weights = pd.Series(dict(optimal_portfolio))
equal_weights = pd.Series(dict(equal_weight_portfolio))

opt_returns, opt_value, opt_end, opt_daily_std, opt_annual_std, opt_annual_return, opt_sharpe = pf_test.calculate_rebalanced_buy_and_hold(
    df=target_returns_df,
    weights=optimal_weights,
    tickers=tickers,
    initial_capital=1_000_000,
    risk_free_rate=0.03
)

eq_returns, eq_value, eq_end, eq_daily_std, eq_annual_std, eq_annual_return, eq_sharpe = pf_test.calculate_rebalanced_buy_and_hold(
    df=target_returns_df,
    weights=equal_weights,
    tickers=tickers,
    initial_capital=1_000_000,
    risk_free_rate=0.03
)

print(f"Optimal ending value: ${opt_end:,.2f}")
print(f"Optimal annual return: {opt_annual_return:.2%}")
print(f"Optimal daily std: {opt_daily_std:.4%}")
print(f"Optimal annualized std: {opt_annual_std:.2%}")
print(f"Optimal Sharpe ratio: {opt_sharpe:.3f}")

print()

print(f"Equal-weight ending value: ${eq_end:,.2f}")
print(f"Equal-weight annual return: {eq_annual_return:.2%}")
print(f"Equal-weight daily std: {eq_daily_std:.4%}")
print(f"Equal-weight annualized std: {eq_annual_std:.2%}")
print(f"Equal-weight Sharpe ratio: {eq_sharpe:.3f}")

Optimal ending value: $2,068,535.33
Optimal annual return: 15.73%
Optimal daily std: 1.5747%
Optimal annualized std: 25.00%
Optimal Sharpe ratio: 0.509

Equal-weight ending value: $1,503,203.18
Equal-weight annual return: 8.54%
Equal-weight daily std: 0.9750%
Equal-weight annualized std: 15.48%
Equal-weight Sharpe ratio: 0.358


In [13]:
final_signal_export_df = pd.read_csv(BASE_URL + "final_signal_export.csv")

#Alternative reproduction path
#final_signal_export_df = pd.read_csv(DATA_DIR / "final_signal_export.csv")

final_signal_export_df

,Unnamed: 0,Date,Ticker,open_price,close_price,combined_signal
0,0,2016-04-01,AAPL,24.636544,24.684103,1
1,1,2016-04-04,AAPL,25.007968,24.910585,1
2,2,2016-04-05,AAPL,24.801878,25.166506,1
3,3,2016-04-06,AAPL,24.964938,24.869822,1
4,4,2016-04-07,AAPL,24.901522,25.130268,1
...,...,...,...,...,...,...
75205,75205,2026-03-16,XOM,156.000000,156.119995,0
75206,75206,2026-03-17,XOM,158.250000,157.229996,0
75207,75207,2026-03-18,XOM,159.660004,158.809998,0
75208,75208,2026-03-19,XOM,158.259995,157.589996,0


In [14]:
final_signal_export_df['combined_signal'].value_counts()

combined_signal
 0    70169
-1     3224
 1     1817
Name: count, dtype: int64

In [15]:
df = final_signal_export_df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df[df['Date'] >= '2021-04-01']
df = df.sort_values(["Ticker", "Date"])

df["prev_signal"] = df.groupby("Ticker")["combined_signal"].shift(1)
df["prev_close"] = df.groupby("Ticker")["close_price"].shift(1)

df = df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

daily_df, trade_log_df, final_positions, final_positions_df, final_portfolio_value = pf_test.backtest_signal_strategy(
    df,
    initial_capital=1_000_000,
    annual_cash_rate=0.03,
    buy_cost_multiplier=1.01,
    sell_cost_multiplier=0.99,
    max_position_weight=0.15,
)

print("Final portfolio value:", round(final_portfolio_value, 2))
print(final_positions_df.sort_values("MarketValue", ascending=False))

Final portfolio value: 1343196.69
  Ticker       Shares   LastClose    MarketValue    Weight
0   CASH          NaN    1.000000  552622.055316  0.411423
1   NVDA  1144.196662  178.559998  204307.753104  0.152106
2    DIS  2052.507401   99.199997  203608.727867  0.151585
3    BAC  4295.620744   47.009998  201937.123945  0.150341
4   MSFT   464.554617  389.019989  180721.032045  0.134545


In [16]:
trade_log_df

,Date,Ticker,Action,SignalUsed,Shares,TradePrice,TradeValue
0,2021-04-28,VZ,BUY,1.0,3545.461428,42.370246,150222.072915
1,2021-04-29,VZ,BUY,1.0,18.351116,42.505540,780.024119
2,2021-05-17,AMD,BUY,1.0,2015.743092,74.962201,151104.539326
3,2021-05-18,AMD,BUY,1.0,3.924571,75.487398,296.255666
4,2021-05-19,AMD,BUY,1.0,39.543077,73.891604,2921.901360
...,...,...,...,...,...,...,...
140,2026-01-21,MSFT,BUY,1.0,76.006854,456.085013,34665.586939
141,2026-02-03,T,SELL,-1.0,8589.367620,25.848901,222025.709870
142,2026-02-13,AAPL,SELL,-1.0,845.567910,259.389910,219331.783906
143,2026-03-17,DIS,BUY,1.0,605.750886,100.161698,60673.037353


In [17]:
total_commission = (trade_log_df["TradeValue"] * 0.01).sum()

print(f"Total transaction cost: ${total_commission:,.2f}")

Total transaction cost: $154,711.93


In [18]:
daily_df

,Date,BeginningCash,CashInterest,Cash,HoldingsValue,TotalPortfolioValue,NumOpenPositions,ActiveBuySignals,DynamicTargetWeight,DailyReturn
0,2021-04-01,1.000000e+06,82.191781,1.000082e+06,0.000000,1.000000e+06,0,0,0.00,0.000000
1,2021-04-05,1.000082e+06,82.198536,1.000164e+06,0.000000,1.000082e+06,0,0,0.00,0.000082
2,2021-04-06,1.000164e+06,82.205292,1.000247e+06,0.000000,1.000164e+06,0,0,0.00,0.000082
3,2021-04-07,1.000247e+06,82.212049,1.000329e+06,0.000000,1.000247e+06,0,0,0.00,0.000082
4,2021-04-08,1.000329e+06,82.218806,1.000411e+06,0.000000,1.000329e+06,0,0,0.00,0.000082
...,...,...,...,...,...,...,...,...,...,...
1243,2026-03-16,6.151082e+05,50.556841,6.151588e+05,732305.140263,1.347413e+06,4,0,0.00,-0.006011
1244,2026-03-17,5.544858e+05,45.574171,5.545313e+05,798072.786176,1.352559e+06,4,1,0.15,0.003819
1245,2026-03-18,5.545313e+05,45.577917,5.545769e+05,800623.493222,1.355155e+06,4,1,0.15,0.001920
1246,2026-03-19,5.525312e+05,45.413525,5.525766e+05,793645.137466,1.346176e+06,4,1,0.15,-0.006625


In [19]:
avg_cash = daily_df["Cash"].mean()

print(f"Average cash: ${avg_cash:,.2f}")

daily_df["CashWeight"] = daily_df["Cash"] / daily_df["TotalPortfolioValue"]

avg_cash_weight = daily_df["CashWeight"].mean()

print(f"Average cash weight: {avg_cash_weight:.2%}")

avg_invested_weight = 1 - avg_cash_weight

print(f"Average invested weight: {avg_invested_weight:.2%}")

print(daily_df["CashWeight"].describe())

Average cash: $366,576.91
Average cash weight: 32.69%
Average invested weight: 67.31%
count    1248.000000
mean        0.326887
std         0.350162
min         0.000000
25%         0.000000
50%         0.158054
75%         0.633008
max         1.000082
Name: CashWeight, dtype: float64


In [21]:
# 1) target weights to test
target_weights = np.array([0.033, 0.05, 0.10, 0.15, 0.20, 0.25, 0.32])
risk_free_rate = 0.03
summary_rows = []

# optional: keep full outputs for each run if you want to inspect them later
all_runs = {}

for weight in target_weights:
    daily_df, trade_log_df, final_positions, final_positions_df, final_portfolio_value = pf_test.backtest_signal_strategy(
        df,
        initial_capital=1_000_000,
        annual_cash_rate=risk_free_rate,
        buy_cost_multiplier=1.01,
        sell_cost_multiplier=0.99,
        max_position_weight=weight,
        min_trade_dollars=1000,
    )

    # --- basic period stats ---
    start_value = 1_000_000
    n_days = len(daily_df)
    n_years = n_days / 252  # trading-day convention

    total_return = final_portfolio_value / start_value - 1
    annual_return = (final_portfolio_value / start_value) ** (1 / n_years) - 1 if n_years > 0 else np.nan

    # --- trading / commission ---
    num_trades = len(trade_log_df)
    total_commission = (trade_log_df["TradeValue"] * 0.01).sum() if not trade_log_df.empty else 0.0

    # --- cash / exposure ---
    avg_cash = daily_df["Cash"].mean()
    daily_df = daily_df.copy()
    daily_df["CashWeight"] = daily_df["Cash"] / daily_df["TotalPortfolioValue"]
    avg_cash_weight = daily_df["CashWeight"].mean()
    avg_invested_weight = 1 - avg_cash_weight

    # --- portfolio volatility based on portfolio daily returns ---
    portfolio_daily_std = daily_df["DailyReturn"].std()

    # optional annualized vol
    annualized_vol = portfolio_daily_std * np.sqrt(252)

    # --- cash interest ---
    total_cash_interest = daily_df["CashInterest"].sum()

    # --- store row ---
    summary_rows.append({
        "MaxPositionWeight": weight,
        "FinalPortfolioValue": final_portfolio_value,
        "TotalReturn": total_return,
        "AnnualReturn": annual_return,
        "NumTrades": num_trades,
        "TotalCommission": total_commission,
        "AverageCash": avg_cash,
        "AverageCashWeight": avg_cash_weight,
        "AverageInvestedWeight": avg_invested_weight,
        "PortfolioDailyStd": portfolio_daily_std,
        "AnnualizedVolatility": annualized_vol,
        "TotalCashInterest": total_cash_interest,
    })

    # optional: keep outputs
    all_runs[weight] = {
        "daily_df": daily_df,
        "trade_log_df": trade_log_df,
        "final_positions": final_positions,
        "final_positions_df": final_positions_df,
        "final_portfolio_value": final_portfolio_value,
    }

summary_df = pd.DataFrame(summary_rows).sort_values("MaxPositionWeight").reset_index(drop=True)

summary_df["SharpeRatio"] = np.where(
    summary_df["AnnualizedVolatility"] > 0,
    (summary_df["AnnualReturn"] - risk_free_rate) / summary_df["AnnualizedVolatility"],
    np.nan
)

summary_df["Sharpe_ExCash"] = (
    (summary_df["AnnualReturn"] - risk_free_rate)
    / summary_df["AnnualizedVolatility"]
) / summary_df["AverageInvestedWeight"]

summary_df

,MaxPositionWeight,FinalPortfolioValue,TotalReturn,AnnualReturn,NumTrades,TotalCommission,AverageCash,AverageCashWeight,AverageInvestedWeight,PortfolioDailyStd,AnnualizedVolatility,TotalCashInterest,SharpeRatio,Sharpe_ExCash
0,0.033,1.250001e+06,0.250001,0.046088,221,61316.558087,837572.973019,0.755901,0.244099,0.002891,0.045886,85907.273676,0.350618,1.436373
1,0.050,1.325481e+06,0.325481,0.058547,247,95702.119306,715175.050940,0.631237,0.368763,0.004384,0.069588,73353.296735,0.410223,1.112431
2,0.100,1.261555e+06,0.261555,0.048034,187,129043.807661,444661.603272,0.411881,0.588119,0.006883,0.109262,45607.567670,0.165051,0.280642
3,0.150,1.343843e+06,0.343843,0.061492,136,154714.759042,366685.588939,0.326964,0.673036,0.008594,0.136424,37609.808646,0.230835,0.342975
4,0.200,1.305242e+06,0.305242,0.055263,95,151894.487606,304732.013621,0.276916,0.723084,0.009445,0.149942,31255.421719,0.168484,0.233007
5,0.250,1.269922e+06,0.269922,0.049434,83,151341.206037,277542.445418,0.252574,0.747426,0.010469,0.166194,28466.671661,0.116933,0.156447
6,0.320,1.389072e+06,0.389072,0.068610,70,167224.443261,262332.203728,0.234597,0.765403,0.011988,0.190306,26906.604135,0.202886,0.265071


#### Discussion of Portfolio Outcomes

The empirical results reveal a clear distinction between signal-level effectiveness and portfolio-level performance.

At the signal level, evidence suggests that the engineered indicators possess a degree of predictive power. This is most visible in low-exposure configurations, where the strategy selectively deploys capital. In these cases, the exposure-adjusted Sharpe ratio exceeds 1.0, indicating that the signals are capable of identifying favorable trading opportunities when applied conservatively.

However, this signal quality does not translate into strong portfolio performance when scaled. As position limits increase and the strategy becomes more fully invested, both absolute and risk-adjusted returns deteriorate. This suggests that the signals exhibit limited scalability, generating value primarily in a sparse and selective context rather than as a broad allocation mechanism.

A significant factor contributing to this outcome is portfolio construction inefficiency. The strategy relies on equal allocation across active signals, without incorporating measures of signal strength, volatility, or conviction. As a result, capital is deployed suboptimally, diluting the impact of higher-quality signals.

In addition, transaction costs materially reduce performance. The strategy exhibits high turnover, leading to substantial cumulative trading costs, which erode a meaningful portion of the gross returns. This effect becomes more pronounced at higher exposure levels, where increased trading activity offsets the benefits of greater capital deployment.

Another important observation is the role of cash allocation. At lower position weights, a large portion of capital remains uninvested, resulting in lower overall returns but artificially enhanced risk-adjusted metrics due to reduced volatility. Conversely, higher exposure increases returns but reveals the underlying limitations of the signal framework.

When compared to benchmark portfolios, the signal-based strategy does not outperform an optimized portfolio constructed using mean-variance principles, and only marginally competes with an equal-weight portfolio on a risk-adjusted basis. This indicates that, in its current form, the strategy is better interpreted as a timing or filtering mechanism, rather than a standalone investment solution.

Overall, the results suggest that while the signals contain some informational value, portfolio construction, execution costs, and exposure management dominate realized performance. Future improvements would likely require enhancements in position sizing, turnover reduction, and integration with a baseline allocation framework.